In [1]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

from transformers import PreTrainedTokenizerFast

In [2]:
from tqdm import tqdm

import datasets 
from datasets import Dataset, DatasetDict, Features, Value, load_dataset, load_from_disk, concatenate_datasets

from LearningEvaluation.data_management import *

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [3]:
data_dir = 'LearningEvaluation/original_training_data/new_sample/mini'
sample_datasets = load_datasets_list(data_dir=data_dir)
data = load_dataset(data_dir=sample_datasets[1])
data

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 50
    })
})

In [4]:
sample_datasets[1]

'LearningEvaluation/original_training_data/new_sample/mini/n=000050'

In [5]:
def batch_iterator(dataset, batch_size=10):
    for i in range(0, len(dataset), batch_size):
        yield dataset[i:i + batch_size]["text"]

In [6]:
special_tokens = [
    "[PAD]",
    "[UNK]",
    "[CLS]",
    "[SEP]",
    "[MASK]",
]

base_tokenizer = Tokenizer(
    BPE(unk_token="[UNK]")
)

base_tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=6000,
    special_tokens=special_tokens,
)

base_tokenizer.train_from_iterator(
    batch_iterator(data["train"]),
    trainer=trainer,
)

In [7]:
print(base_tokenizer.get_vocab_size()) #== 8061

6000


In [8]:
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=base_tokenizer,
    unk_token="<UNK>",
    pad_token="<PAD>",
    bos_token="<BOS>",
    sep_token="<SEP>",
    mask_token="<MASK>"
)
tokenizer.padding_side = "left"

In [9]:
tokenizer.pad_token_id

6003

In [10]:
print("vocab_size:", tokenizer.vocab_size)
print("len(tokenizer):", len(tokenizer))

print(tokenizer.special_tokens_map)

vocab_size: 6000
len(tokenizer): 6005
{'bos_token': '<BOS>', 'unk_token': '<UNK>', 'sep_token': '<SEP>', 'pad_token': '<PAD>', 'mask_token': '<MASK>'}


In [11]:
def test(tokenizer, data):
    encoding = tokenizer(
        data["train"][0:10]["text"],
        return_tensors="pt",
        padding=True
    )
    
    print(data["train"][0]["text"])
    print(encoding)
    print(tokenizer.convert_ids_to_tokens(
        encoding["input_ids"][0]
    ))

test(tokenizer, data)

Don Walsh (1932-2006) was a football player who mainly played for the Saskatchewan Roughriders in the Canadian Football League (CFL) from 1953 to 1964 at offensive guard and linebacker.

Don Walsh was born in Trois-Rivières, Quebec, and grew up in Montreal. Walsh played college football at the University of Denver. Walsh first joined the Calgary Stampeders in 1953 but played most of his career as a member of the Saskatchewan Roughriders from 1955 to 1964 at the positions of offensive guard and linebacker. 

After studying the subject at the University of Denver and the University of Arizona and after his professional career, Walsh became an architect. He died from cancer in Vancouver at the age of 74.

References

Category:1932 births
Category:2006 deaths
Category:Canadian football linebackers
Category:Canadian football offensive linemen
Category:Players of Canadian football from Quebec
Category:Sportspeople from Trois-Rivières
Category:Canadian players of American football
Category:De

In [12]:
from MeMoHF.modelling_memo_trainable_tokenizer import TrainedMeMoTokenizer

tokenizer = TrainedMeMoTokenizer.train(data['train'], truncation_side='left', 
                                       padding_side='left', model_max_length=4096)
tokenizer

TrainedMeMoTokenizer(name_or_path='', vocab_size=8061, model_max_length=4097, is_fast=True, padding_side='left', truncation_side='left', special_tokens={'bos_token': '<BOS>', 'unk_token': '<UNK>', 'sep_token': '<SEP>', 'pad_token': '<PAD>', 'mask_token': '<MASK>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("<UNK>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<PAD>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<BOS>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<SEP>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<MASK>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [20]:
a = tokenizer.get_text_batch_encoding("1,2,3,4,5,6,7")
print(a)
tokenizer.decode(a['input_ids'][0], skip_special_tokens=False)

{'input_ids': tensor([[ 1,  1,  1,  ..., 13, 23, 13]]), 'labels': tensor([[ 1,  1,  1,  ..., 23, 13, 24]])}


'<PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PA